# Cross-task failure memory on HotpotQA (DS 340W, Group 48)

Extends Reflexion (Shinn et al., 2023). The agent runs the Reflexion loop on **training** questions and
reformulates each failure as a lesson (failure type + diagnosis + general fix). A lesson is kept only if it
(1) repairs the question it came from on a fresh attempt and (2) helps more **development** questions than it
breaks. Kept lessons are retrieved **before the first attempt** on unseen **test** questions.

Conditions: `no_memory`, `reflexion` (parent paper), `failure_memory` (proposed), `random_memory`,
`similarity_memory`, `retry_memory` (lessons checked by retry only), `unvalidated_memory` (no check).

**Setup:** add your Groq key in Colab Secrets (key icon on the left) as `GROQ_API_KEY`.

In [ ]:
!git clone -q https://github.com/omaraljashmi/reflexion-hotpotqa-reproduction.git
%cd reflexion-hotpotqa-reproduction
!pip -q install -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
MODEL = "openai/gpt-oss-20b"   # try a different Groq model if the pilot is outside 40-75%
COMMON = "--level hard --context gold --workers 8 --tpm 235000"   # lower --tpm on the free tier

## 0. Offline tests (no API calls)

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## 1. Pilot: is the model making enough mistakes to learn from?
Target first-attempt exact match: roughly 40-75%.

In [ ]:
!python run_experiment.py pilot --provider groq --model $MODEL --n 40 $COMMON

## 2. Build the lesson memory from training questions
Reflexion on 1,000 lesson questions, then both checks. 100 separate development questions test each lesson.

In [ ]:
!python run_experiment.py build --provider groq --model $MODEL --n 1000 --dev-n 100 $COMMON

In [ ]:
import json
bank = json.load(open("results/memory.json"))
print(json.dumps(bank["meta"]["funnel"], indent=1))
for l in [l for l in bank["lessons"] if l["validated"]][:10]:
    print(f'{l["id"]} [{l["qtype"]}/{l["failure_type"]}] helped {l["dev_helped"]}, hurt {l["dev_hurt"]} of {l["dev_questions"]}')
    print(f'  diagnosis: {l["diagnosis"]}\n  fix: {l["fix"]}\n')

## 3. Evaluate on unseen test questions

In [ ]:
!python run_experiment.py evaluate --provider groq --model $MODEL --n 300 --trials 4 --conditions all --label rules $COMMON
# repeat the four planned conditions on the same questions to see how stable the numbers are
!python run_experiment.py evaluate --provider groq --model $MODEL --n 300 --trials 4 --repeats 2 --resume --label rules $COMMON

## 4. Analysis (reruns from the saved CSV, no API key needed)

In [ ]:
from IPython.display import Image, Markdown, display
!python run_experiment.py analyze --attempts results/eval_attempts.csv | tail -25
display(Markdown(open("results/comparison_table.md").read()))
display(Image("results/comparison.png", width=700))

In [ ]:
import pandas as pd
df = pd.read_csv("results/eval_attempts.csv")
first = df[(df.trial == 1) & (df.run_id == 0)]
display(first.groupby(["condition", "qtype"]).exact_match.mean().unstack().round(3))